# Introduction

This Notebook demostrates how to fine-tune BERT for a text classification task.

The dataset used is **AG News**, included with `datasets` library.

The dataset includes the following categories:

    0: "World"
    1: "Sports"
    2: "Business"
    3: "Sci/Tech"

In the dataset there are 120,000 samples in train, and 7,600 in test.

We will subsample 5K from train and 500 from test.

# Import libraries

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict, Dataset
from sklearn.metrics import accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
model_name = "bert-base-uncased"


# Load and tokenize the data

In [ ]:
# subsample the relatively large dataset
def stratified_sample_hf(ds, n_samples):
    df = ds.to_pandas()
    n_classes = df["label"].nunique()
    per_class = n_samples // n_classes

    df_sampled = (
        df.groupby("label")
          .apply(lambda x: x.sample(n=per_class, random_state=42))
          .reset_index(drop=True)
    )

    return Dataset.from_pandas(df_sampled)

In [ ]:
dataset = load_dataset("ag_news")
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

train_small = stratified_sample_hf(dataset["train"], 5000)
test_small = stratified_sample_hf(dataset["test"], 1000)

dataset = DatasetDict({
    "train": train_small,
    "test": test_small
})

dataset = dataset.map(tokenize, batched=True)
dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Prepare the model

In [ ]:
# initialize the model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
)

# define compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

# training arguments initialization
training_args = TrainingArguments(
    output_dir="./bert-ag-news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
)


# Train the model

In [ ]:
trainer.train()

# Evaluate the model

In [ ]:
trainer.evaluate()

# Prediction with the model

In [ ]:
import torch

id_to_label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}

def predict(text):
    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128,
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits

    return id_to_label[logits.argmax(dim=-1).item()]

print(predict("Apple announced a new chip for artificial intelligence workloads."))

